In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
model.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.CIFAR10(
    root="./data/cifar10",
    train=False,
    download=True,
    transform=transform
)

print(len(dataset))

In [ ]:
img, label = dataset[0]

inputs = processor(images=img, return_tensors="pt").to(device)

with torch.no_grad():
    emb = model.get_image_features(**inputs)

print(emb.shape)
print(label)

In [ ]:
embeddings = []
labels = []

for i in range(200):
    img, label = dataset[i]

    inputs = processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        emb = model.get_image_features(**inputs)

    embeddings.append(emb.cpu().numpy())
    labels.append(label)

embeddings = np.vstack(embeddings)

print(embeddings.shape)

In [ ]:
import faiss

index = faiss.IndexFlatIP(512)
index.add(embeddings.astype("float32"))

query = embeddings[0:1]
D, I = index.search(query.astype("float32"), k=5)

print(I)

In [ ]:
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
images, labels = next(iter(dataloader))
print(images.shape)